In [ ]:
import numpy as np

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction
        
class MyDecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def gini(self, y):
        if len(y) == 0:
            return 0
        proportions=np.unique(y, return_counts=True)[1]/len(y)
        return 1-np.sum(proportions**2)

    def split_gain(self, y, left_y, right_y):
        parent_impurity = self.gini(y)
        weighted_children = (
            len(left_y) / len(y) * self.gini(left_y)
            + len(right_y) / len(y) * self.gini(right_y)
        )
        return parent_impurity - weighted_children

    def make_split(self, X, y):
        parent_gini=self.gini(y)
        best_gain=0.0
        best_split=None

        features = X.shape[1]

        for feature in range(features):
            values=np.unique(X[:, feature])
            thresholds=(values[:-1] + values[1:])/2

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = X[:, feature] > threshold
                if not left_mask.any() or not right_mask.any():
                    continue

                left_y = y[left_mask]
                right_y = y[right_mask]

                gain=self.split_gain(y,left_y,right_y)

                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        "feature": feature,
                        "threshold": threshold,
                        "gain": gain,
                        "left_mask": left_mask,
                        "right_mask": right_mask,
                    }

        return best_split
        
    def make_leaf(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return Node(prediction=classes[np.argmax(counts)])

    def build_tree(self, X, y, depth=0):
        if (len(np.unique(y)) == 1 or depth >= self.max_depth or len(y) < self.min_samples_split):
            return self.make_leaf(y)

        split = self.make_split(X, y)

        if split is None:
            return self.make_leaf(y)

        left_mask = split["left_mask"]
        right_mask = split["right_mask"]

        return Node(
            feature=split["feature"],
            threshold=split["threshold"],
            left=self.build_tree(X[left_mask], y[left_mask], depth + 1),
            right=self.build_tree(X[right_mask], y[right_mask], depth + 1)
        )
        
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.root=self.build_tree(X, y)
        return self

    def predict_one(self, x):
        node=self.root
        while node.prediction is None:
            if x[node.feature] <= node.threshold:
                node=node.left
            else:
                node=node.right
        return node.prediction

    def predict(self, X):
        X = np.asarray(X)
        return np.array([self.predict_one(row) for row in X])

    def score(self, X, y):
        y=np.asarray(y)
        predictions=self.predict(X)
        return np.mean(predictions==y)


In [2]:
import pandas as pd
import numpy as np

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

df = pd.read_csv(
    "processed.cleveland.data",
    header=None,
    names=columns,
    na_values="?"
).dropna()

categorical_features = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"
]

X = pd.get_dummies(
    df.drop(columns="num"),
    columns=categorical_features,
    dtype=int
).to_numpy(dtype=float)

y = df["num"].to_numpy(dtype=int)

# Shuffle and make an 80:20 train/test split
rng = np.random.default_rng(42)
indices = rng.permutation(len(y))
split_at = int(0.8 * len(y))

train_indices = indices[:split_at]
test_indices = indices[split_at:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

tree = MyDecisionTree(max_depth=5, min_samples_split=2)
tree.fit(X_train, y_train)

print("Test accuracy:", tree.score(X_test, y_test))

Test accuracy: 0.55
